---
## 13. Question F - Original research question

**1. Question.** *How persistent are listings from one day to the next within our daily samples, and are the prices of listings that appear repeatedly stable?*

**2. Why it is relevant.** Persistence shows how fast the visible marketplace turns over and whether the seven daily files can be treated as the same population or as different samples. Price stability tells us whether price changes over a week are a real market feature. Both are directly needed to interpret Question C. They were chosen after seeing the data, because the identifier and the repeated collection are the strongest parts of the dataset (unlike condition for Fashion or location for Furniture).

**3. Variables.** `listing_key` (from the URL), `collection_date`, `category`, `price`.

**4. Method.** Count on how many different days each unique listing appears; compute the overlap between days for Mobile Phones (the only category collected on all seven days); compare first and last prices of repeated listings.

**5. Visualisations.** F1 heatmap of day-to-day overlap (phones) and F2 distribution of the number of days each listing was seen, by category.

In [1]:
# Q F: How persistent are listings in the daily samples, and are repeated listings' prices stable?
days_seen = df.groupby("listing_key").agg(category=("category", "last"), n_days=("collection_date", "nunique"))
dist = days_seen.groupby(["category", "n_days"]).size().unstack(fill_value=0).reindex(CATS)
dist_pct = dist.div(dist.sum(axis=1), axis=0) * 100
display(dist)
print("Share of each category's unique listings seen on exactly 1 day, %:"); print(dist_pct[1].round(1).to_string())
print("Share seen on 2 or more days, %:"); print((100 - dist_pct[1]).round(1).to_string())

# Overlap between days, Mobile Phones only (the only category collected on all 7 days)
ph_sets = {d: set(g.listing_key) for d, g in df[df.category == "Mobile Phones"].groupby("collection_date")}
dl = sorted(ph_sets)
overlap = pd.DataFrame([[100 * len(ph_sets[a] & ph_sets[b]) / len(ph_sets[a]) for b in dl] for a in dl],
                       index=[d.strftime("%d %b") for d in dl], columns=[d.strftime("%d %b") for d in dl])
display(overlap.round(1))
cons = [100 * len(ph_sets[a] & ph_sets[b]) / len(ph_sets[a]) for a, b in zip(dl[:-1], dl[1:])]
print("Phones: % of one day's listings that are also present the NEXT collected day:", [round(x, 1) for x in cons])

# Price stability of repeated listings
stab = matched.assign(changed=lambda d: d.direction != "No change")
print("\nShare of repeated listings whose price changed, by number of days seen:")
print((stab.groupby("n_days")["changed"].agg(["mean", "size"]).rename(columns={"mean": "share_changed", "size": "listings"}).assign(share_changed=lambda d: (100 * d.share_changed).round(1))).to_string())
print("Share changed by span between first and last observation (days):")
bins = pd.cut(stab["span_days"], [0, 2, 4, 6, 7], labels=["1-2 d", "3-4 d", "5-6 d", "7 d"])
print(stab.groupby(bins)["changed"].agg(["mean", "size"]).assign(mean=lambda d: (100 * d["mean"]).round(1)).to_string())

NameError: name 'df' is not defined

**Findings.** 78.5% of unique phone listings, 86.8% of furniture listings and 76.1% of fashion listings were seen on **only one** of the seven days (so 21.5%, 13.2% and 23.9% were seen on two or more days). Of one day's phone listings, only 15.5%, 9.0%, 27.0%, 29.5%, 9.0% and 7.5% were present on the next collected day, and by 19 October just 4% of the 13 October phone listings remained. Repeated listings kept their price: 0.6% of those seen on 2 days changed price, 0% of those on 3, 4 or 6 days, and 2 of 13 (15.4%) of those seen on 5 days; by span between first and last sighting, 0.3% (1-2 days), 0.5% (3-4 days) and 3.4% (5-6 days; 2 of 59) changed.

In [ ]:
# Chart F1: Phones - % of the row-day's listings that are also present on the column day
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(overlap, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100, linewidths=.5, cbar_kws={"label": "% of row-day listings also seen on column day"}, ax=ax)
ax.set_xlabel("Column day (2026)"); ax.set_ylabel("Row day (2026)")
ax.set_title("F1. Mobile Phones: how many listings re-appear on another day?")
plt.tight_layout(); plt.show()

**What the chart shows:** For Mobile Phones, each cell is the percentage of the row day's listings that also appear on the column day (diagonal = 100%).

**Main finding:** Overlap is highest between 15, 16 and 17 October (27 - 29.5%) and between 13 and 15/17 October (25%, 19%); it is lowest with 19 October (3 - 8.5%) and 18 October (6.5 - 9%). The 13 and 19 October phone samples share only 4% of listings.

**Interpretation:** The visible set turns over quickly: from one day to the next, about 70 - 90% of phone listings are new to the sample. Overlap falls with distance in time, as expected for ads being posted, sold or pushed down the results.

**Limitation:** The sample is the first 200 results per day, so a listing that "disappears" may still be live on Jiji, just beyond the first results page(s). Cells involving 14 and 18 October rest on equal-sized but incomplete runs.

**Answer to the research question:** Persistence is low and declines with time.

In [ ]:
# Chart F2: number of different days on which each listing was seen, by category
fig, ax = plt.subplots(figsize=(10, 5))
w = .26; x = np.arange(1, 7)
for k, c in enumerate(CATS):
    vals = [dist.loc[c].get(i, 0) for i in x]
    bars = ax.bar(x + (k - 1) * w, [100 * v / dist.loc[c].sum() for v in vals], w, color=PAL[c], label=f"{c} (n={int(dist.loc[c].sum()):,} listings)")
    for b, v in zip(bars, vals):
        if v: ax.text(b.get_x() + b.get_width() / 2, b.get_height() + .8, str(v), ha="center", fontsize=7.5)
ax.set_xlabel("Number of different collection days on which the listing appeared"); ax.set_ylabel("% of the category's unique listings")
ax.set_title("F2. Listing persistence: on how many of the 7 days was each listing seen? (labels = number of listings)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

**What the chart shows:** For each category, the percentage (bar) and number (label) of unique listings seen on 1, 2, ..., 6 different collection days. No listing was seen on all 7 days.

**Main finding:** Most listings appear only once: 822 of 1,047 phone, 891 of 1,027 furniture and 536 of 704 fashion listings. Fashion has the most repeated listings (23.9% seen on 2+ days; 30 listings on 4 days, 7 on 5) and Furniture the fewest (13.2%). The maximum is 6 days (4 phone listings).

**Interpretation:** Fashion adverts persist longest in the sample; Furniture turns over fastest. Together with price stability, this shows that listings that remain visible rarely change their price.

**Limitation:** Fashion was collected on 5 days and Furniture on 6 of 7, so those categories had fewer chances to be re-sighted than Mobile Phones; the comparison across categories is therefore not fully fair.

**Answer to the research question:** Listings are short-lived in the sample (most are seen once) and, when seen repeatedly, their prices are stable.

**Overall answer to Question F, and limitations.** About three-quarters of listings are seen on a single day and only 19% on two or more; repeated listings keep their price in 99.2% of cases (525 of 529). Limitations: sampling of the first 200 results, unequal daily coverage, and the fact that disappearance from the sample does not prove that the ad was removed.

---
## 14. Final automatic validation

The following cell re-checks the key facts of the data pipeline (row counts, categories, dates, prices, duplicates, posting dates and the rebuilt database).

In [ ]:
# Final automatic consistency checks (validation before submission)
checks = {
 "7 daily files loaded": raw["source_file"].nunique() == 7,
 "combined rows = sum of file rows": len(raw) == dq["rows"].sum() == len(clean),
 "category counts agree with raw": clean["category"].value_counts().to_dict() == raw["category"].str.strip().str.title().value_counts().to_dict(),
 "collection_date agrees with file date for all rows": bool(clean["date_matches_file"].all()),
 "all prices numeric & positive": bool(clean["price_valid"].all()),
 "no exact duplicate rows": exact_dups == 0,
 "posting_date missing in every row": bool(raw["posting_date"].isna().all()),
 "rebuilt DB row count equals CSV rows": n_obs_rb == len(clean),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values()), "A consistency check failed"
print("All checks passed. Raw CSV files were only read, never written.")

---
## 15. Final Answers to the Assignment Research Questions

| Assignment question | Data and method used | Main finding | Evidence / visualisation | Limitations | Final answer |
|---|---|---|---|---|---|
| **A. Price distribution by category; outliers** | 3,600 valid prices; summary statistics, 1.5 x IQR rule, box plots and log-bin histograms | Medians: Phones 35,000; Furniture 9,999; Fashion 1,700 KSh. All right-skewed. High outliers only: 149 / 95 / 84 observations | Charts A1, A2; statistics table; outlier inspection | Pooled repeated observations; non-random sample of first 200 results/day | **Answered.** Phones highest median and most outliers; Fashion lowest median; outliers mostly plausible, except one KSh 400,350 t-shirt |
| **C. Changes in prices, discounts, rankings; biggest movers** | Daily medians; matched listings by listing code (529 listings seen 2+ days) | Median prices fluctuate with no trend (changing sample); only 4 of 529 matched listings changed price (3 up, 1 down) | Charts C1-C4; matched-listing tables | No old price/discount/rank fields; unequal daily coverage; Fashion/Furniture missing on some days | **Partially answered.** Prices: yes (stable). Discounts and rankings: **not answerable**. Biggest movers: unisex clothes listing (+153%), Poco M8 Pro (+4.7%), Galaxy S26 Ultra (+0.8%), recliner seats (-12.5%) |
| **D. Dominant brands or sellers** | Rule-based brand extraction from titles; unique-listing counts; product-type keywords for other categories | Phones: Samsung 36.1% and Apple 27.5% of unique listings; brands not identifiable in Furniture (0.3%) / Fashion (2.0%) | Charts D1, D2 (alternative for D2) | Title-based; no seller data (prohibited) | **Partially answered.** Brands for phones only; seller dominance **not answerable** |
| **F. Original question: listing persistence and price stability** | Listing code across days; overlap matrix; first vs last price | 78.5% / 86.8% / 76.1% of phone / furniture / fashion listings seen on 1 day only; 99.2% of repeated listings unchanged in price | Charts F1, F2 | First-200 sample; unequal coverage | **Answered.** Listings turn over quickly and rarely change price |

---
## 16. Key Findings and Conclusions

**Price patterns.** Categories live in different price worlds: median KSh 35,000 for Mobile Phones, 9,999 for Furniture and 1,700 for Fashion. All distributions are right-skewed, so medians describe typical prices better than means. Outliers are all on the high side and mostly plausible (flagship phones, large furniture, premium jackets); one KSh 400,350 t-shirt listing is probably an entry error or bulk price.

**Condition.** Brand New dominates (82.8% of observations). Used items are not cheaper in the raw comparison (phones: used 40,000 vs new 33,999.5; furniture: 25,000 vs 9,500), mostly because of what is sold in each group. Refurbished phones have the lowest median (27,999) and the smallest spread.

**Time.** Daily medians fluctuate without a trend because the sample changes: only 7.5 - 29.5% of a day's phone listings reappear the next day. Among 529 matched listings, 525 kept the same price (3 increases, 1 decrease), so advertised prices are very sticky.

**Brands and locations.** Samsung (36.1%) and Apple (27.5%) of unique phone listings dominate Mobile Phones; brands cannot be identified reliably in Furniture or Fashion. 95.4% of observations are in Nairobi county (Nairobi Central 2,596 observations).

**Original question.** Most listings are seen on only one day (76 - 87% depending on category), and repeated listings seldom change price.

**Data-quality problems to disclose.**
1. `scraped_at` in the 13 October file is corrupt (599 of 600 values); only the date is usable.
2. `advert_id` is the full URL, not a separate id; listing codes (`listing_key`) were extracted.
3. `posting_date` is empty in all 3,600 records.
4. 14 October (Furniture, Fashion) and 18 October (Fashion) were not collected because of connection errors.
5. The SQLite database lacks all of 13 October and contains 600 extra 16 October rows; it was rebuilt from the CSVs.
6. Condition contains the non-condition value "Unisex" (8 Fashion rows).
7. No rating, review count, rank, old price or discount fields.

**Questions limited by missing fields.** B (rating/review/rank), the discount and ranking parts of C, the seller part of D, and E. Alternative analyses were clearly labelled and do not replace them.

**Recommendations for future collection.** Record old price, discount %, rating/review count where displayed, and the position of each listing in the results; use a unique run id and write timestamps with Python's `datetime.isoformat()` (avoiding a row-counter in the offset); retry failed category pages and log a per-run completeness report; keep the database loader idempotent (unique on listing code + run date); capture the posting date from the advert page.

**Conclusion.** The seven-day Jiji Kenya dataset supports a solid analysis of price distributions, outliers, brand concentration in phones and listing turnover. It does not support analysis of ratings, rankings or discounts because those fields were never collected, and its daily samples differ in size and coverage, so no market-wide price trend can be claimed. Matched listings show that advertised prices on Jiji are rarely changed within a week, while the marketplace's visible listings turn over quickly.